# Checkpoint 03: Tiền Xử Lý và Chuẩn Hóa Dữ Liệu Mạng Xã Hội (Data Preprocessing)

Nghiên cứu: Nâng cao hiệu năng mô hình phát hiện phát ngôn thù ghét trên mạng xã hội tiếng Việt (ViHSD).

Mục tiêu khoa học:
1. Khắc phục hạn chế của pipeline tiền xử lý gốc (vốn loại bỏ stopwords bừa bãi và dùng deEmojify tiếng Anh gây nhiễu).
2. Xây dựng lớp xử lý chuyên biệt `ViHSDPreprocessor` tích hợp trực tiếp trong notebook:
   - Làm sạch nhiễu URL, ký tự điều khiển, chuẩn hóa khoảng trắng.
   - Ánh xạ biểu cảm emoji 2 pha (giữ trọn sắc thái thù ghét/xúc phạm) qua từ điển chuyên dụng `emojis.csv`.
   - Chuẩn hóa tiếng lóng, viết tắt mạng xã hội qua từ điển `teencode.csv` (412 quy tắc chuẩn).
   - Thu gọn ký tự kéo dài về 1 ký tự (`ngonnnn` -> `ngon`).
   - Phân đoạn từ tiếng Việt tự động bằng thư viện `pyvi` theo cấu trúc từ ghép của PhoBERT (`ViTokenizer.tokenize`).
   - **Bảo tồn tuyệt đối** đại từ nhân xưng (`mày`, `tao`, `nó`, `thằng`, `con`) và từ phủ định (`không`, `chẳng`, `chả`).
3. Đánh giá tác động độc lập của bước tiền xử lý trên cấu hình Baseline gốc (`vinai/phobert-base`, 2 epochs, Cross-Entropy Loss tiêu chuẩn).
4. Xuất đầy đủ 4 tệp Mandatory Artifacts theo chuẩn vào thư mục `outputs/checkpoint03/`.

Ghi chú: Toàn bộ mã nguồn `ViHSDPreprocessor`, `BuildDataset` và hàm đánh giá được tích hợp 100% trong notebook để bảo đảm khả năng chạy độc lập trên Google Colab.

In [ ]:
import os
import sys

# Thiết lập môi trường làm việc và kết nối Google Drive trên Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    project_dir = '/content/drive/MyDrive/NLP_PhoBert_ViHSD'
except ImportError:
    project_dir = os.path.abspath('..')

checkpoints_dir = os.path.join(project_dir, 'checkpoints', 'checkpoint03')
outputs_dir = os.path.join(project_dir, 'outputs', 'checkpoint03')

os.makedirs(checkpoints_dir, exist_ok=True)
os.makedirs(outputs_dir, exist_ok=True)

print(f'Thư mục dự án: {project_dir}')
print(f'Thư mục lưu checkpoint: {checkpoints_dir}')
print(f'Thư mục lưu kết quả: {outputs_dir}')

In [ ]:
# Cài đặt các thư viện cần thiết trên môi trường Google Colab
!pip install -q pyvi transformers accelerate scikit-learn seaborn matplotlib pandas numpy torch

In [ ]:
# Thiết lập hạt giống ngẫu nhiên cố định để đảm bảo tính tái lập thực nghiệm
random_seed = 42

import os
import random
import numpy as np
import torch

os.environ['PYTHONHASHSEED'] = str(random_seed)
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(random_seed)
    torch.cuda.manual_seed_all(random_seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f'Thiết lập random seed thành công: {random_seed}')

## 1. Nạp Tập Dữ Liệu Thô và Kiểm Tra Trước Xử Lý (Pre-pipeline Verification)

In [ ]:
# Cấu hình đường dẫn tập dữ liệu
data_dir = 'data' if os.path.exists('data/train.csv') else os.path.join(project_dir, 'data')
train_path = os.path.join(data_dir, 'train.csv')
dev_path = os.path.join(data_dir, 'dev.csv')
test_path = os.path.join(data_dir, 'test.csv')

import pandas as pd

train_raw = pd.read_csv(train_path)
dev_raw = pd.read_csv(dev_path)
test_raw = pd.read_csv(test_path)

assert len(train_raw) == 24048, f'Số lượng mẫu train không khớp: {len(train_raw)}'
assert len(dev_raw) == 2672, f'Số lượng mẫu dev không khớp: {len(dev_raw)}'
assert len(test_raw) == 6680, f'Số lượng mẫu test không khớp: {len(test_raw)}'

print('Kiểm tra tập dữ liệu thô trước xử lý:')
print(f'- Train: {len(train_raw)} mẫu, null free_text: {train_raw["free_text"].isna().sum()}')
print(f'- Dev:   {len(dev_raw)} mẫu, null free_text: {dev_raw["free_text"].isna().sum()}')
print(f'- Test:  {len(test_raw)} mẫu, null free_text: {test_raw["free_text"].isna().sum()}')

## 2. Định Nghĩa Bộ Tiền Xử Lý Dữ Liệu Chuyên Sâu (ViHSDPreprocessor)

Lớp xử lý văn bản tiếng Việt mạng xã hội tự chứa, giải quyết các đặc thù tiếng lóng, emoji và phân đoạn từ.

In [ ]:
import re
import unicodedata
import numpy as np
import pandas as pd

try:
    from pyvi import ViTokenizer
except ImportError:
    ViTokenizer = None


class ViHSDPreprocessor:

    def __init__(self, teencode_path, emoji_path, segmenter=None):
        self.segmenter = segmenter
        self.teencode_dict = self._load_teencode_dict(teencode_path)
        self.emoji_dict = self._load_emoji_dict(emoji_path)

        self.url_re          = re.compile(r'https?://\S+|www\.\S+')
        self.ws_re           = re.compile(r'\s+')
        self.ctrl_re         = re.compile(r'[\r\n\t]+')
        self.lengthen_re     = re.compile(r'([a-zA-Z\u00C0-\u1EF9])\1{2,}', re.IGNORECASE)

        tc_keys = [re.escape(k) for k in sorted(self.teencode_dict, key=len, reverse=True)]
        self.tc_re = re.compile(r'\b(' + '|'.join(tc_keys) + r')\b', re.IGNORECASE) if tc_keys else None

        em_keys = [re.escape(k) for k in sorted(self.emoji_dict, key=len, reverse=True)]
        self.em_re = re.compile('(' + '|'.join(em_keys) + ')') if em_keys else None

    def _load_teencode_dict(self, path):
        df = pd.read_csv(path)
        k = 'teencode'     if 'teencode'     in df.columns else df.columns[0]
        v = 'replacement'  if 'replacement'  in df.columns else ('standard' if 'standard' in df.columns else df.columns[1])
        return dict(zip(df[k].astype(str).str.strip().str.lower(),
                        df[v].astype(str).str.strip().str.lower()))

    def _load_emoji_dict(self, path):
        df = pd.read_csv(path)
        k = 'emoji'       if 'emoji'       in df.columns else df.columns[0]
        v = 'replacement' if 'replacement' in df.columns else ('meaning' if 'meaning' in df.columns else df.columns[1])
        return dict(zip(df[k].astype(str).str.strip(),
                        df[v].astype(str).str.strip()))

    def _clean(self, text):
        text = unicodedata.normalize('NFC', text)
        text = self.url_re.sub(' ', text)
        text = self.ctrl_re.sub(' ', text)
        return self.ws_re.sub(' ', text).strip()

    def _map_emojis(self, text):
        if self.em_re is None:
            return text
        text = self.em_re.sub(lambda m: f' {self.emoji_dict.get(m.group(0), "")} ', text)
        return self.ws_re.sub(' ', text).strip()

    def _norm_teencode(self, text):
        if self.tc_re is None:
            return text
        return self.tc_re.sub(lambda m: self.teencode_dict.get(m.group(0).lower(), m.group(0)), text)

    def _reduce_lengthen(self, text):
        return self.lengthen_re.sub(r'\1', text)

    def _segment(self, text):
        if self.segmenter is not None:
            try:
                out = self.segmenter.word_segment(text)
                return ' '.join(out).strip() if isinstance(out, list) else str(out).strip()
            except Exception:
                pass
        return ViTokenizer.tokenize(text) if ViTokenizer else text

    def transform_text(self, text, lowercased=False):
        if not isinstance(text, str):
            text = str(text) if text is not None else ''
        if not text.strip():
            return ''
        text = self._clean(text)
        text = self._map_emojis(text)
        text = self._norm_teencode(text)
        text = self._reduce_lengthen(text)
        if text.strip():
            text = self._segment(text)
        if lowercased:
            text = text.lower()
        return self.ws_re.sub(' ', text).strip()

    def transform_dataset(self, df, text_col='free_text', label_col='label_id', lowercased=False):
        df = df.dropna(subset=[text_col, label_col]).copy()
        df[label_col] = df[label_col].astype(int)
        texts, indices = [], []
        for idx, row in df.iterrows():
            t = self.transform_text(row[text_col], lowercased=lowercased)
            if t:
                texts.append(t)
                indices.append(idx)
        return texts, df.loc[indices, label_col].to_numpy()

    @staticmethod
    def _norm_key(text):
        """Chuẩn hóa để so khớp trùng lặp: NFC + lowercase + bỏ dấu câu + chuẩn hóa khoảng trắng."""
        if not isinstance(text, str):
            text = str(text) if text is not None else ''
        text = unicodedata.normalize('NFC', text).lower()
        text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
        text = re.sub(r'[^\w\s]', ' ', text)
        return re.sub(r'\s+', ' ', text).strip()

    @classmethod
    def clean_and_deduplicate(cls, train_df, dev_df=None, test_df=None,
                               text_col='free_text', label_col='label_id',
                               drop_conflicting=True, verbose=True):
        """Loại bỏ data leakage, label conflict và duplicate khỏi tập train."""
        df = train_df.dropna(subset=[text_col, label_col]).copy()
        df[label_col] = df[label_col].astype(int)
        df['_k'] = df[text_col].apply(cls._norm_key)
        n0 = len(df)

        # 1. Loại bỏ leakage (train ∩ dev/test)
        eval_keys = set()
        if dev_df is not None:
            eval_keys |= set(dev_df[text_col].dropna().apply(cls._norm_key))
        if test_df is not None:
            eval_keys |= set(test_df[text_col].dropna().apply(cls._norm_key))
        df = df[~df['_k'].isin(eval_keys)]
        n1 = len(df)

        # 2. Loại bỏ label conflict (cùng văn bản, khác nhãn)
        conflict_keys = set(
            df.groupby('_k')[label_col].nunique().pipe(lambda s: s[s > 1]).index
        )
        if drop_conflicting:
            df = df[~df['_k'].isin(conflict_keys)]
        else:
            df = df.drop_duplicates(subset=['_k'], keep='first')
        n2 = len(df)

        # 3. Khử trùng lặp nội bộ
        df = df.drop_duplicates(subset=['_k'], keep='first')
        n3 = len(df)

        if verbose:
            print(f'  Leakage removed   : {n0 - n1:>5,}  ({n0:,} → {n1:,})')
            print(f'  Conflict removed  : {n1 - n2:>5,}  ({n1:,} → {n2:,})')
            print(f'  Duplicate removed : {n2 - n3:>5,}  ({n2:,} → {n3:,})')
            print(f'  Final train size  : {n3:>5,}')

            label_names = {0: 'clean', 1: 'offensive', 2: 'hate'}
            before = train_df.dropna(subset=[label_col])[label_col].astype(int).value_counts().sort_index()
            after  = df[label_col].value_counts().sort_index()
            print(f'  {"Label":<12} {"Before":>8} {"After":>8} {"Δ":>7}')
            for lbl, name in label_names.items():
                b, a = before.get(lbl, 0), after.get(lbl, 0)
                print(f'  {name:<12} {b:>8,} {a:>8,} {a-b:>+7,}')

        return df.drop(columns=['_k'])

    @classmethod
    def verify_no_leakage(cls, train_df, dev_df=None, test_df=None,
                           text_col='free_text'):
        """Assert rằng train không còn overlap với dev/test và không có duplicate nội bộ."""
        train_keys = set(train_df[text_col].dropna().apply(cls._norm_key))

        if dev_df is not None:
            overlap = train_keys & set(dev_df[text_col].dropna().apply(cls._norm_key))
            assert not overlap, f'Leakage: {len(overlap)} mẫu train còn trong Dev'

        if test_df is not None:
            overlap = train_keys & set(test_df[text_col].dropna().apply(cls._norm_key))
            assert not overlap, f'Leakage: {len(overlap)} mẫu train còn trong Test'

        _df = train_df[text_col].dropna().apply(cls._norm_key)
        assert not _df.duplicated().any(), 'Còn duplicate nội bộ trong train'

        print('Không còn leakage và duplicate trong tập train.')


## 3. Khử Trùng Lặp, Triệt Tiêu Rò Rỉ Dữ Liệu (Data Leakage) và Tiền Xử Lý Văn Bản

Khảo sát và xử lý các vấn đề chất lượng dữ liệu cốt lõi trong tập ViHSD:
1. **Triệt tiêu rò rỉ dữ liệu (Data Leakage Elimination):** Loại bỏ hoàn toàn các câu trong tập Train có mặt trong tập Dev hoặc Test để đảm bảo mô hình không học vẹt dữ liệu đánh giá, bảo toàn tính khách quan trong đánh giá tổng quát hóa.
2. **Khử nhiễu nhãn mâu thuẫn (Label Noise Resolution):** Loại bỏ các văn bản giống hệt nhau nhưng bị gán nhiều nhãn khác nhau (0, 1, 2) do bất đồng giữa các người gán nhãn, ngăn ngừa gradient đối nghịch làm mất ổn định hàm mất mát.
3. **Khử trùng lặp nội bộ (Deduplication):** Giữ lại duy nhất một bản ghi đại diện cho các mẫu trùng lặp cùng nhãn để tránh thiên lệch tần suất.
4. **Chuẩn hóa ngôn ngữ mạng xã hội:** Áp dụng  (Teencode 412 quy tắc, Emoji 2 pha, thu gọn lặp ký tự và phân đoạn từ PyVi).

In [ ]:
teencode_path = os.path.join(data_dir, 'teencode.csv')
emoji_path    = os.path.join(data_dir, 'emojis.csv')

# 1. Khử trùng lặp và loại bỏ data leakage khỏi tập train
print('=== Deduplication & Leakage Elimination ===')
train_clean_df = ViHSDPreprocessor.clean_and_deduplicate(
    train_df=train_raw, dev_df=dev_raw, test_df=test_raw,
    text_col='free_text', label_col='label_id',
    drop_conflicting=True, verbose=True
)

# 2. Xác minh không còn leakage
ViHSDPreprocessor.verify_no_leakage(
    train_df=train_clean_df, dev_df=dev_raw, test_df=test_raw
)

# 3. Tiền xử lý văn bản trên ba tập
preprocessor = ViHSDPreprocessor(teencode_path=teencode_path, emoji_path=emoji_path)
print('\n=== Preprocessing ===')

train_X, train_y = preprocessor.transform_dataset(train_clean_df)
dev_X,   dev_y   = preprocessor.transform_dataset(dev_raw)
test_X,  test_y  = preprocessor.transform_dataset(test_raw)

assert len(train_X) == len(train_y)
assert len(dev_X)   == len(dev_y)
assert len(test_X)  == len(test_y)
assert all(t.strip() for t in train_X), 'Tồn tại văn bản rỗng trong train_X'

print(f'  Train : {len(train_X):,} mẫu')
print(f'  Dev   : {len(dev_X):,} mẫu')
print(f'  Test  : {len(test_X):,} mẫu')


## 4. Định Nghĩa Lớp BuildDataset và Mã Hóa Bằng PhoBERT FastBPE (max_length = 100)

In [ ]:
pretrained_model_name = 'vinai/phobert-base'
max_seq_length = 100

import torch
from transformers import AutoTokenizer


class BuildDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)


print(f'Khởi tạo Tokenizer: {pretrained_model_name} (use_fast=False, max_length={max_seq_length})')
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name, use_fast=False)

train_encodings = tokenizer(train_X, truncation=True, padding=True, max_length=max_seq_length)
dev_encodings = tokenizer(dev_X, truncation=True, padding=True, max_length=max_seq_length)
test_encodings = tokenizer(test_X, truncation=True, padding=True, max_length=max_seq_length)

train_dataset = BuildDataset(train_encodings, train_y)
dev_dataset = BuildDataset(dev_encodings, dev_y)
test_dataset = BuildDataset(test_encodings, test_y)

print(f'Kích thước tập train sau mã hóa: {len(train_dataset)}')
print(f'Kích thước tập dev sau mã hóa:   {len(dev_dataset)}')
print(f'Kích thước tập test sau mã hóa:  {len(test_dataset)}')

## 5. Cấu Hình Huấn Luyện Cơ Sở và Hàm Mất Mát Tiêu Chuẩn

Giữ nguyên hàm mất mát Cross-Entropy tiêu chuẩn để cô lập chính xác mức độ đóng góp của bước tiền xử lý chuyên sâu `ViHSDPreprocessor`.

In [ ]:
num_labels = 3
num_epochs = 2
train_batch_size = 16
eval_batch_size = 16
learning_rate = 2e-5
warmup_steps = 500
weight_decay = 0.01

import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments


def compute_metrics_eval(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=-1)
    acc = accuracy_score(labels, preds)
    macro_p = precision_score(labels, preds, average='macro', zero_division=0)
    macro_r = recall_score(labels, preds, average='macro', zero_division=0)
    macro_f1 = f1_score(labels, preds, average='macro')
    micro_f1 = f1_score(labels, preds, average='micro')
    per_class_f1 = f1_score(labels, preds, average=None)
    return {
        'accuracy': acc,
        'macro_precision': macro_p,
        'macro_recall': macro_r,
        'macro_f1': macro_f1,
        'micro_f1': micro_f1,
        'f1_clean': per_class_f1[0] if len(per_class_f1) > 0 else 0.0,
        'f1_offensive': per_class_f1[1] if len(per_class_f1) > 1 else 0.0,
        'f1_hate': per_class_f1[2] if len(per_class_f1) > 2 else 0.0,
    }


model = AutoModelForSequenceClassification.from_pretrained(pretrained_model_name, num_labels=num_labels)

training_args = TrainingArguments(
    output_dir=checkpoints_dir,
    num_train_epochs=num_epochs,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,
    warmup_steps=warmup_steps,
    weight_decay=weight_decay,
    learning_rate=learning_rate,
    do_eval=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=False,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    seed=random_seed,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics_eval,
)

print('Khởi tạo Trainer thành công cho Checkpoint 03.')

## 6. Huấn Luyện Mô Hình và Lưu Checkpoint

In [ ]:
print('Bắt đầu huấn luyện Checkpoint 03 (Baseline + Preprocessing)...')
train_result = trainer.train()

save_model_dir = os.path.join(checkpoints_dir, 'final_model')
trainer.save_model(save_model_dir)
tokenizer.save_pretrained(save_model_dir)
print(f'Đã lưu mô hình tại: {save_model_dir}')

## 7. Đánh Giá Trên Tập Test Độc Lập

In [ ]:
class_names = ['clean', 'offensive', 'hate']

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score


def evaluate_predictions(y_true, y_pred, class_names=None):
    if class_names is None:
        class_names = ['clean', 'offensive', 'hate']
    acc = accuracy_score(y_true, y_pred)
    macro_precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    macro_recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    micro_f1 = f1_score(y_true, y_pred, average='micro')
    per_class_precision = precision_score(y_true, y_pred, average=None, zero_division=0)
    per_class_recall = recall_score(y_true, y_pred, average=None, zero_division=0)
    per_class_f1 = f1_score(y_true, y_pred, average=None)
    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
    return {
        'accuracy': acc,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'micro_f1': micro_f1,
        'per_class_precision': {name: per_class_precision[i] for i, name in enumerate(class_names)},
        'per_class_recall': {name: per_class_recall[i] for i, name in enumerate(class_names)},
        'per_class_f1': {name: per_class_f1[i] for i, name in enumerate(class_names)},
        'confusion_matrix': cm,
        'classification_report': report,
    }


print('Thực hiện dự đoán với mô hình Checkpoint 03 trên tập kiểm tra...')
predictions_output = trainer.predict(test_dataset)
y_pred = np.argmax(predictions_output.predictions, axis=-1)
y_true = test_y

results = evaluate_predictions(y_true, y_pred, class_names=class_names)

print('=' * 60)
print('KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST (CHECKPOINT 03)')
print('=' * 60)
print(f'Accuracy (Độ chính xác tổng thể):  {results["accuracy"]:.4f}')
print(f'Macro-Precision (Độ chuẩn xác):   {results["macro_precision"]:.4f}')
print(f'Macro-Recall (Độ nhạy):           {results["macro_recall"]:.4f}')
print(f'Macro-F1 Score (Chỉ số chính):    {results["macro_f1"]:.4f}')
print(f'Micro-F1 Score:                   {results["micro_f1"]:.4f}')
print('-' * 60)
print('Chi tiết theo từng lớp (Per-class):')
for cname in class_names:
    print(f'- {cname:9s}: Precision = {results["per_class_precision"][cname]:.4f}, Recall = {results["per_class_recall"][cname]:.4f}, F1 = {results["per_class_f1"][cname]:.4f}')
print('=' * 60)
print('Chi tiết Classification Report:')
print(results['classification_report'])

## 8. Phân Tích Lỗi Định Tính và Trực Quan Hóa Ma Trận Nhầm Lẫn

In [ ]:
cm_output_path = os.path.join(outputs_dir, 'confusion_matrix_checkpoint03.png')

import matplotlib.pyplot as plt
import seaborn as sns


def plot_confusion_matrix(cm, class_names=None, output_path=None):
    if class_names is None:
        class_names = ['clean', 'offensive', 'hate']
    df_cm = pd.DataFrame(cm, index=class_names, columns=class_names)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(df_cm, annot=True, cmap='Blues', fmt='d', cbar=True, ax=ax, annot_kws={'size': 12})
    ax.set_title('Confusion Matrix - Checkpoint 03 (Data Preprocessing)')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=300)
    return fig


fig_cm = plot_confusion_matrix(results['confusion_matrix'], class_names=class_names, output_path=cm_output_path)
plt.show()
print(f'Đã lưu ma trận nhầm lẫn tại: {cm_output_path}')

## 9. Xuất Các Tệp Kết Quả Chuẩn Hóa (Mandatory Artifacts Export)

In [ ]:
predictions_csv_path = os.path.join(outputs_dir, 'predictions.csv')
scores_csv_path = os.path.join(outputs_dir, 'evaluation_scores.csv')
report_csv_path = os.path.join(outputs_dir, 'classification_report.csv')
cm_csv_path = os.path.join(outputs_dir, 'confusion_matrix.csv')

test_ids = list(range(len(test_y)))

df_predictions = pd.DataFrame({
    'id': test_ids,
    'text': test_X,
    'ground_truth': y_true,
    'predicted_label': y_pred,
})
df_predictions.to_csv(predictions_csv_path, index=False, encoding='utf-8')
print(f'1. Đã xuất tệp dự đoán: {predictions_csv_path}')

df_cm = pd.DataFrame(results['confusion_matrix'], index=class_names, columns=class_names)
df_cm.index.name = 'true_label'
df_cm.to_csv(cm_csv_path)
print(f'2. Đã xuất tệp dữ liệu ma trận nhầm lẫn: {cm_csv_path}')

acc_val = round(float(results.get('accuracy', accuracy_score(y_true, y_pred))), 4)
macro_p_val = round(float(results.get('macro_precision', precision_score(y_true, y_pred, average='macro', zero_division=0))), 4)
macro_r_val = round(float(results.get('macro_recall', recall_score(y_true, y_pred, average='macro', zero_division=0))), 4)
per_class_p = results.get('per_class_precision', {name: round(precision_score(y_true, y_pred, average=None, zero_division=0)[i], 4) for i, name in enumerate(class_names)})
per_class_r = results.get('per_class_recall', {name: round(recall_score(y_true, y_pred, average=None, zero_division=0)[i], 4) for i, name in enumerate(class_names)})

scores_summary = {
    'metric': [
        'accuracy',
        'macro_precision',
        'macro_recall',
        'macro_f1',
        'micro_f1',
        'precision_clean',
        'recall_clean',
        'f1_clean',
        'precision_offensive',
        'recall_offensive',
        'f1_offensive',
        'precision_hate',
        'recall_hate',
        'f1_hate',
    ],
    'value': [
        acc_val,
        macro_p_val,
        macro_r_val,
        round(results['macro_f1'], 4),
        round(results['micro_f1'], 4),
        round(per_class_p['clean'], 4),
        round(per_class_r['clean'], 4),
        round(results['per_class_f1']['clean'], 4),
        round(per_class_p['offensive'], 4),
        round(per_class_r['offensive'], 4),
        round(results['per_class_f1']['offensive'], 4),
        round(per_class_p['hate'], 4),
        round(per_class_r['hate'], 4),
        round(results['per_class_f1']['hate'], 4),
    ],
}
df_scores = pd.DataFrame(scores_summary)
df_scores.to_csv(scores_csv_path, index=False)
print(f'3. Đã xuất tệp điểm số tổng hợp: {scores_csv_path}')

report_dict = classification_report(y_true, y_pred, target_names=class_names, digits=4, output_dict=True)
accuracy_val = report_dict.pop('accuracy')

rows = []
for label in class_names:
    row = report_dict[label]
    rows.append({
        'class': label,
        'precision': round(row['precision'], 4),
        'recall': round(row['recall'], 4),
        'f1_score': round(row['f1-score'], 4),
        'support': int(row['support']),
    })

rows.append({
    'class': 'accuracy',
    'precision': None,
    'recall': None,
    'f1_score': round(accuracy_val, 4),
    'support': len(y_true),
})

for avg_type in ['macro avg', 'weighted avg']:
    row = report_dict[avg_type]
    rows.append({
        'class': avg_type,
        'precision': round(row['precision'], 4),
        'recall': round(row['recall'], 4),
        'f1_score': round(row['f1-score'], 4),
        'support': int(row['support']),
    })

df_report = pd.DataFrame(rows)
df_report.to_csv(report_csv_path, index=False)
print(f'4. Đã xuất tệp báo cáo phân loại chi tiết: {report_csv_path}')